# 01 — Data Profiling
Project FORESIGHT — Week 1, Day 2-3

**Goal:** inspect each of the 4 raw files to find data-quality issues *before* cleaning anything.
We are not fixing anything in this notebook — only documenting what's wrong.


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

DATA_DIR = '../data'

sales_daily = pd.read_csv(f'{DATA_DIR}/sales_daily.csv', parse_dates=['date'])
sku_master = pd.read_csv(f'{DATA_DIR}/sku_master.csv', parse_dates=['launch_date'])
calendar = pd.read_csv(f'{DATA_DIR}/calendar.csv', parse_dates=['date'])
inventory_snapshots = pd.read_csv(f'{DATA_DIR}/inventory_snapshots.csv', parse_dates=['date'])

datasets = {
    'sales_daily': sales_daily,
    'sku_master': sku_master,
    'calendar': calendar,
    'inventory_snapshots': inventory_snapshots,
}
print("Loaded 4 datasets.")

Loaded 4 datasets.


## 1. Shape & data types
How many rows/columns does each file have, and are the types what we expect?

In [2]:
for name, df in datasets.items():
    print(f"--- {name} ---")
    print(f"Shape: {df.shape}")
    print(df.dtypes)
    print()

--- sales_daily ---
Shape: (129517, 6)
date          datetime64[us]
sku_id                   str
units_sold             int64
revenue              float64
unit_price           float64
promo_flag             int64
dtype: object

--- sku_master ---
Shape: (200, 6)
sku_id                    str
category                  str
subcategory               str
launch_date    datetime64[us]
unit_cost             float64
list_price            float64
dtype: object

--- calendar ---
Shape: (731, 6)
date           datetime64[us]
week                    int64
month                   int64
season                    str
is_holiday              int64
promo_event               str
dtype: object

--- inventory_snapshots ---
Shape: (18533, 6)
date              datetime64[us]
sku_id                       str
on_hand_units              int64
on_order_units             int64
lead_time_days           float64
reorder_point              int64
dtype: object



## 2. Preview each dataset
Just eyeball the first few rows of each.

In [3]:
for name, df in datasets.items():
    print(f"--- {name} ---")
    display(df.head(3))

--- sales_daily ---


,date,sku_id,units_sold,revenue,unit_price,promo_flag
0,2024-12-07,SKU0011,10,18164.90,1816.49,0
1,2024-10-19,SKU0031,14,34807.36,2486.24,0
2,2025-07-22,SKU0141,8,14235.76,1779.47,0


--- sku_master ---


,sku_id,category,subcategory,launch_date,unit_cost,list_price
0,SKU0001,Decor,Wall Art,2024-01-01,3151.81,6105.58
1,SKU0002,Decor,Candles,2024-01-01,3484.78,6763.97
2,SKU0003,Textiles,Throws,2024-01-01,2929.70,6576.29


--- calendar ---


,date,week,month,season,is_holiday,promo_event
0,2024-01-01,1,1,Winter,1,NaN
1,2024-01-02,1,1,Winter,0,NaN
2,2024-01-03,1,1,Winter,0,NaN


--- inventory_snapshots ---


,date,sku_id,on_hand_units,on_order_units,lead_time_days,reorder_point
0,2024-01-01,SKU0001,168,242,14.0,132
1,2024-01-08,SKU0001,156,144,14.0,132
2,2024-01-15,SKU0001,201,256,14.0,132


## 3. Missing values
Count and percentage of missing values per column.

In [4]:
for name, df in datasets.items():
    missing = df.isna().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    report = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
    report = report[report['missing_count'] > 0]
    print(f"--- {name} ---")
    if report.empty:
        print("No missing values.")
    else:
        display(report)
    print()

--- sales_daily ---


,missing_count,missing_pct
revenue,1295,1.00
unit_price,1316,1.02



--- sku_master ---
No missing values.

--- calendar ---


,missing_count,missing_pct
promo_event,647,88.51



--- inventory_snapshots ---


,missing_count,missing_pct
lead_time_days,197,1.06


## 4. Duplicate rows
Exact duplicate rows shouldn't exist in a clean dataset.

In [5]:
for name, df in datasets.items():
    dupes = df.duplicated().sum()
    print(f"{name}: {dupes} duplicate rows ({dupes/len(df)*100:.2f}%)")

sales_daily: 387 duplicate rows (0.30%)
sku_master: 0 duplicate rows (0.00%)
calendar: 0 duplicate rows (0.00%)
inventory_snapshots: 0 duplicate rows (0.00%)


## 5. Sanity checks on values
Look for impossible values: negative sales, negative prices, dates out of range, etc.

In [6]:
# sales_daily checks
print("Negative units_sold rows:", (sales_daily['units_sold'] < 0).sum())
print("Negative revenue rows:", (sales_daily['revenue'] < 0).sum())
print("Date range:", sales_daily['date'].min(), "to", sales_daily['date'].max())
print()

# sku_master checks
print("Unique sku_id count:", sku_master['sku_id'].nunique(), "vs total rows:", len(sku_master))
print("Negative or zero unit_cost:", (sku_master['unit_cost'] <= 0).sum())
print("list_price < unit_cost (loss-making):", (sku_master['list_price'] < sku_master['unit_cost']).sum())

Negative units_sold rows: 132
Negative revenue rows: 0
Date range: 2024-01-01 00:00:00 to 2025-12-31 00:00:00

Unique sku_id count: 200 vs total rows: 200
Negative or zero unit_cost: 0
list_price < unit_cost (loss-making): 0


## 6. Inconsistent category labels
Check for label inconsistencies (casing, spelling variants) — a very common real-world issue.

In [7]:
print("Unique category values in sku_master:")
print(sku_master['category'].unique())
print()
print("Value counts:")
print(sku_master['category'].value_counts())

Unique category values in sku_master:
<StringArray>
['Decor', 'Textiles', 'Furniture', 'Lighting', 'Small Appliances']
Length: 5, dtype: str

Value counts:
category
Furniture           56
Decor               48
Small Appliances    43
Lighting            30
Textiles            23
Name: count, dtype: int64


## 7. Referential integrity
Do sku_id values in sales_daily and inventory_snapshots all exist in sku_master? Do dates in sales_daily all exist in calendar?

In [8]:
valid_skus = set(sku_master['sku_id'])
orphan_sales = ~sales_daily['sku_id'].isin(valid_skus)
orphan_inv = ~inventory_snapshots['sku_id'].isin(valid_skus)
print("sales_daily rows with unknown sku_id:", orphan_sales.sum())
print("inventory_snapshots rows with unknown sku_id:", orphan_inv.sum())

valid_dates = set(calendar['date'])
orphan_dates = ~sales_daily['date'].isin(valid_dates)
print("sales_daily rows with dates missing from calendar:", orphan_dates.sum())

sales_daily rows with unknown sku_id: 0
inventory_snapshots rows with unknown sku_id: 0
sales_daily rows with dates missing from calendar: 0


## 8. Summary of findings (fill this in as you go)

Write down what you found here — this becomes the core of your **data-quality report (D2)**.

**Example structure:**
- `sales_daily`: X% missing `revenue`, X% missing `unit_price`, X duplicate rows, X rows with negative `units_sold`
- `sku_master`: category labels inconsistent (mixed casing) — needs standardizing
- `inventory_snapshots`: X% missing `lead_time_days`
- No orphan SKUs or dates found — referential integrity is clean

**Next step:** use these findings to write the cleaning logic in `src/pipeline.py` (Week 1, Day 4-5).
